# Laser diode efficiencies {#sec-ld-efficiency}

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/philippevelha/Optoelectronics-teaching/blob/main/chapter4/notebooks/ld-efficiency.ipynb){.colab-link} &nbsp;Run this chapter's code in Google Colab (no installation).

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/philippevelha/Optoelectronics-teaching/blob/main/chapter4/notebooks/ld-efficiency.ipynb)

> Generated from the Quarto companion. The browser sliders of the website are not included; see `interactive_explorers.ipynb` for ipywidgets versions.

In [ ]:
# --- setup: works locally and in Google Colab ---------------------------
import sys
if 'google.colab' in sys.modules:
    import urllib.request
    urllib.request.urlretrieve('https://raw.githubusercontent.com/philippevelha/Optoelectronics-teaching/main/chapter4/laserlib.py', 'laserlib.py')
else:
    sys.path.insert(0, '..')                 # laserlib.py is in the parent folder
def ojs_define(**kw): pass                   # only used by the Quarto website
# ----------------------------------------------------------------------
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import laserlib as lz
lz.use_style()

*Slides 108–117 (slope efficiency, external quantum efficiency, external differential quantum efficiency, internal quantum
efficiency, internal differential quantum efficiency, extraction efficiency, power conversion efficiency, commercial red and
violet laser diodes) and 118–122 (worked examples: a sky-blue laser diode and an InGaAs Fabry–Perot laser).*

## Key relations

| efficiency | definition | slide |
|---|---|---|
| slope $\eta_\text{slope}$ | $\Delta P_o/\Delta I\approx P_o/(I-I_{th})$ (W/A) | 108 |
| external quantum (EQE) | $\eta_\text{EQE}=\dfrac{P_o/h\nu}{I/e}$: output photons per injected electron | 109 |
| external differential (EDQE) | $\eta_\text{EDQE}=\dfrac{\Delta P_o/h\nu}{\Delta I/e}\approx\dfrac{P_o/h\nu}{(I-I_{th})/e}$ | 110 |
| internal quantum (IQE) | $\eta_\text{IQE}=\dfrac{1/\tau_r}{1/\tau_r+1/\tau_{nr}}$ | 111 |
| internal differential (IDQE) | extra stimulated photons generated inside per extra electron above threshold | 112 |
| extraction (EE) | $\eta_\text{EE}=\dfrac{(1/2L)\ln(1/R_1)}{\alpha_t}$: share of the total loss that leaves through facet 1 | 113 |
| power conversion (PCE) | $\eta_\text{PCE}=P_o/(IV)$ | 115 |

: Laser diode efficiencies. {tbl-colwidths="[25,55,20]"}

Above threshold every extra electron becomes a stimulated photon with probability $\eta_\text{IDQE}$ (close to 1), and that photon
leaves through the output facet with probability $\eta_\text{EE}$. Hence (slide 114)

$$
P_o=\eta_\text{EE}\,\eta_\text{IDQE}\,\frac{h\nu}{e}(I-I_{th}),\qquad
\eta_\text{slope}=\eta_\text{EE}\,\eta_\text{IDQE}\,\frac{h\nu}{e},\qquad
\eta_\text{EDQE}=\eta_\text{EE}\,\eta_\text{IDQE}.
$$

The EQE is always smaller than the EDQE, because the threshold current pumps the medium without producing coherent light.

## Laser diode calculator

```{ojs}
//| echo: false
viewof ld = Inputs.form({
  lam: Inputs.range([400, 1700], {value: 1310, step: 1, label: "wavelength λ (nm)"}),
  L: Inputs.range([100, 2000], {value: 200, step: 10, label: "cavity length L (µm)"}),
  as: Inputs.range([1, 50], {value: 20, step: 0.5, label: "internal loss α_s (cm⁻¹)"}),
  R1: Inputs.range([0.01, 0.99], {value: 0.33, step: 0.01, label: "front facet R₁ (output)"}),
  R2: Inputs.range([0.01, 0.99], {value: 0.33, step: 0.01, label: "rear facet R₂"}),
  idqe: Inputs.range([0.3, 1], {value: 1, step: 0.01, label: "internal differential QE η_IDQE"}),
  Ith: Inputs.range([1, 100], {value: 5, step: 0.5, label: "threshold current I_th (mA)"}),
  V0: Inputs.range([0.5, 5], {value: 1.0, step: 0.05, label: "turn-on voltage V₀ (V)"}),
  Rs: Inputs.range([0, 30], {value: 5, step: 0.5, label: "series resistance R_s (Ω)"}),
  I: Inputs.range([0, 200], {value: 20, step: 0.5, label: "operating current I (mA)"})
})
```

```{ojs}
//| echo: false
ldOut = {
  const L = ld.L*1e-4, mir1 = Math.log(1/ld.R1)/(2*L), at = ld.as + Math.log(1/(ld.R1*ld.R2))/(2*L), ee = mir1/at;
  const hv = 1239.84/ld.lam, slope = ee*ld.idqe*hv;                    // W/A
  const curve = [], Imax = Math.max(ld.I*1.5, 3*ld.Ith);
  for (let i = 0; i <= 200; i++) {
    const I = Imax*i/200, P = slope*Math.max(I - ld.Ith, 0), V = ld.V0 + I*1e-3*ld.Rs;
    curve.push({I, P, V, pce: I > 0 ? 100*P/(I*V) : 0});
  }
  const I = ld.I, P = slope*Math.max(I - ld.Ith, 0), V = ld.V0 + I*1e-3*ld.Rs;
  const best = curve.reduce((m, d) => d.pce > m.pce ? d : m, curve[0]);
  return {at, ee, slope, curve, P, V, eqe: P/hv/I, edqe: ee*ld.idqe, pce: P/(I*V), best, hv, rearP: ee > 0 ? P*Math.log(1/ld.R2)/Math.log(1/ld.R1) : 0};
}
Plot.plot({
  width: Math.min(width, 1050), height: 280, x: {label: "current I (mA)"}, y: {label: "P_o (mW) and PCE (%)", grid: true},
  color: {legend: true, domain: ["output power P_o (mW)", "power conversion efficiency (%)"], range: ["#e34948", "#2a78d6"]},
  marks: [Plot.line(ldOut.curve, {x: "I", y: "P", stroke: () => "output power P_o (mW)", strokeWidth: 2}),
          Plot.line(ldOut.curve, {x: "I", y: "pce", stroke: () => "power conversion efficiency (%)", strokeWidth: 2}),
          Plot.ruleX([ld.I], {stroke: "#52514e", strokeDasharray: "3,3"})]
})
```

```{ojs}
//| echo: false
html`<div class="result-box">α<sub>t</sub> = ${ldOut.at.toFixed(1)} cm⁻¹ · η<sub>EE</sub> = <b>${(100*ldOut.ee).toFixed(1)} %</b> · hν = ${ldOut.hv.toFixed(3)} eV ·
η<sub>slope</sub> = <b>${ldOut.slope.toFixed(3)} mW/mA</b><br>
at I = ${ld.I} mA: P<sub>o</sub> = <b>${ldOut.P.toFixed(2)} mW</b> (rear facet ${ldOut.rearP.toFixed(2)} mW), V = ${ldOut.V.toFixed(2)} V ·
EQE = <b>${(100*ldOut.eqe).toFixed(1)} %</b> · EDQE = <b>${(100*ldOut.edqe).toFixed(1)} %</b> · PCE = <b>${(100*ldOut.pce).toFixed(1)} %</b><br>
maximum PCE ${ldOut.best.pce.toFixed(1)} % at ${ldOut.best.I.toFixed(1)} mA</div>`
```

Things to try:

* Coat the rear facet ($R_2=0.95$): almost all the light now leaves through the front, and the slope efficiency rises. Commercial
  edge emitters use a high-reflection rear coating and a 5–30 % front coating.
* Lengthen the cavity: the mirror loss shrinks, so $\eta_\text{EE}$ drops; the threshold also rises (not modelled here, see
  @sec-ld-equation). High-power lasers are long (1–4 mm) to spread the heat, and compensate with very low internal loss.
* The PCE has a maximum: at low current the threshold dominates, at high current the $I^2R_s$ loss.

## Worked examples, recomputed

> **Slides 118–119.** Sky-blue laser diode (Nichia NDS4113), 488 nm, $I_{th}=30$ mA, $P_o=60$ mW at 100 mA and 5.6 V. Find the
> slope efficiency, PCE, EQE and EDQE.
>
> **Slides 120–122.** InGaAs FP laser at 1310 nm, $L=200$ µm, $\alpha_s=20$ cm⁻¹, $R_1=R_2=0.33$, $\eta_\text{IDQE}\approx1$,
> $I_{th}=5$ mA, $V=1.3$ V. Find $P_o$ at 20 mA, the EDQE and the PCE.

In [ ]:
#| label: tbl-ld-examples
#| tbl-cap: "The two laser-diode efficiency examples recomputed."
e1 = lz.ld_efficiencies(60e-3, 0.100, 5.6, 30e-3, 488e-9)
at = lz.threshold_gain(2000, 200e-6, 0.33); ee = lz.extraction_efficiency(2000, 200e-6, 0.33)
hv = lz.h*lz.c0/1310e-9; Po = ee*1.0*hv*(0.020 - 0.005)/lz.qe
rows = [("blue: slope efficiency", f"{e1['slope']:.2f} mW/mA", "0.86 mW/mA"),
        ("blue: PCE", f"{e1['PCE']:.1%}", "11 %"), ("blue: EQE", f"{e1['EQE']:.1%}", "24 %"), ("blue: EDQE", f"{e1['EDQE']:.1%}", "34 %"),
        ("InGaAs: total loss α_t", f"{at:.0f} m⁻¹", "7543 m⁻¹"), ("InGaAs: extraction η_EE", f"{ee:.1%}", "37 %"),
        ("InGaAs: P_o at 20 mA", f"{Po*1e3:.2f} mW", "5.2 mW"), ("InGaAs: slope", f"{Po/0.015:.3f} mW/mA", "0.35 mW/mA"),
        ("InGaAs: EDQE = η_EE η_IDQE", f"{ee:.1%}", "37 %"), ("InGaAs: PCE", f"{Po/(0.020*1.3):.1%}", "20 %")]
pd.DataFrame(rows, columns=["quantity", "computed", "slide"])

All values match the slides. Note that the InGaAs laser emits the same power from its rear facet: the EDQE through *both* facets
is 74 %.

## Commercial red and violet laser diodes (slides 116–117)

In [ ]:
#| label: tbl-commercial
#| tbl-cap: "Slide 116 with recomputed slope (P_o/(I − I_th)), PCE (P_o/IV) and EDQE (slope × e/hν). All rows agree with the slide except the slope of the 500 mW red laser (see text)."
d = pd.DataFrame([("Red", 500, 670, 400, 700, 2.4, 1.0, 30), ("Red", 100, 660, 75, 180, 2.5, 1.0, 22),
                  ("Red", 50, 660, 60, 115, 2.3, 0.90, 19), ("Red", 10, 639, 30, 40, 2.3, 1.0, 11),
                  ("Violet", 400, 405, 160, 390, 5.0, 1.7, 21), ("Violet", 120, 405, 45, 120, 5.0, 1.6, 20),
                  ("Violet", 10, 405, 26, 35, 4.8, 1.1, 6.0)],
                 columns=["LD", "P_o (mW)", "λ (nm)", "I_th (mA)", "I (mA)", "V (V)", "slope, slide", "PCE % slide"])
d["slope = P_o/(I − I_th)"] = (d["P_o (mW)"]/(d["I (mA)"] - d["I_th (mA)"])).round(2)
d["PCE % = P_o/IV"] = (100*d["P_o (mW)"]/(d["I (mA)"]*d["V (V)"])).round(1)
d["EDQE %"] = (100*d["slope = P_o/(I − I_th)"]*d["λ (nm)"]/1239.84).round(0)
d["EQE %"] = (100*d["P_o (mW)"]/d["I (mA)"]*d["λ (nm)"]/1239.84).round(0)
d

For the 500 mW red laser, $P_o/(I-I_{th})=500/300=1.67$ mW/mA, not the 1.0 quoted. Either the threshold or the operating current
on the slide is misprinted, or the quoted slope is the differential slope at high power, reduced by heating. All other rows are
consistent. Violet lasers need about 5 V because of the wide InGaN bandgap and the resistive p-type GaN: their PCE suffers even
though their EDQE is similar.

::: {.callout-tip collapse="true"}
## Going further
- **Measuring the internal loss and IDQE.** For a set of lasers of different lengths, $1/\eta_\text{EDQE}=\frac{1}{\eta_\text{IDQE}}
  \left[1+\frac{2\alpha_sL}{\ln(1/R_1R_2)}\right]$. Plot $1/\eta_\text{EDQE}$ versus $L$: the intercept gives $\eta_\text{IDQE}$, the
  slope $\alpha_s$. Generate synthetic data with the calculator and fit them.
- **Wall-plug efficiency records.** 9xx nm high-power diode bars reach PCE above 70 %. Where do the remaining losses go? Make a
  pie chart: below-threshold current, voltage defect ($eV-h\nu$), series resistance, internal absorption, carrier leakage.
- **Optimum front reflectance.** For a given gain–current relation (@sec-ld-equation), find $R_1$ that maximises the output at a
  given current. The answer differs from the passive extraction optimum because the threshold changes with $R_1$.
:::

::: {.callout-note collapse="true"}
## Complements
- Kasap, sec. 4.12; Coldren *et al.*, sec. 2.5–2.7 (efficiency and the cutback method).
- Datasheets of commercial laser diodes (Thorlabs, Nichia, Ushio) list $I_{th}$, $\eta_\text{slope}$ and $V_F$: check the slide-116
  table against a datasheet of your choice.
:::

## Try it yourself

1. A 980 nm pump laser emits 500 mW at 600 mA and 1.9 V with $I_{th}=50$ mA. Compute the four efficiencies. Why is a high PCE
   essential for pumps of EDFAs (@sec-edfa)?
2. With the calculator, find the front reflectance that gives the highest output at 50 mA when $R_2=0.95$, assuming $I_{th}$ does
   not change. Is the assumption reasonable?
3. Show that $\eta_\text{EDQE}=\eta_\text{slope}\lambda[\mu\text{m}]/1.24$ when the slope is in W/A.
4. Why can the EQE of a laser diode approach the EDQE at high current?